In [ ]:
!pip install deap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 4.2 MB/s eta 0:00:00


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("camnugent/california-housing-prices")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'california-housing-prices' dataset.
Path to dataset files: /kaggle/input/california-housing-prices


In [ ]:
import numpy as np
from deap import base, creator
import random
from deap import tools
import deap
from deap.algorithms import eaSimple
import sklearn
from sklearn.datasets import fetch_california_housing
import time

creator.create("Fitness", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.Fitness)

In [ ]:
california_housing = fetch_california_housing(as_frame=True, return_X_y=True)[0]
x = fetch_california_housing(as_frame=True, return_X_y=True)[0]
y = fetch_california_housing(as_frame=True, return_X_y=True)[1]
california_housing["MedHouseVal"] = fetch_california_housing(as_frame=True, return_X_y=True)[1]

x_train, x_test, y_train, y_test = sklearn.model_selection.train_test_split(x, y, test_size=0.5)
x_train, x_train_validation, y_train, y_train_validation = sklearn.model_selection.train_test_split(x_train, y_train, test_size=0.6)
x_test, x_test_validation, y_test, y_test_validation = sklearn.model_selection.train_test_split(x_train, y_train, test_size=0.6)

In [ ]:
def gerar_hiperparametros():
    hiperparametros = {
        "n_estimators": [10, 30, 50, 100, 200, 500],
        "criterion": ["squared_error", "absolute_error", "friedman_mse", "poisson"],
        "max_depth": [2, 4, 8, 16, 32],
        "min_samples_split": [0.2, 0.1, 0.01, 0.001],
        "min_samples_leaf": [0.2, 0.1, 0.01, 0.001],
        "max_features": ["sqrt", "log2", None]
    }
    conjunto_gerado = []
    for key in hiperparametros.keys():
        i = np.random.randint(len(hiperparametros[key]))
        conjunto_gerado.append(hiperparametros[key][i])
    return conjunto_gerado

toolbox = base.Toolbox()
toolbox.register("individual", tools.initIterate, creator.Individual, gerar_hiperparametros)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate(individuo): # loss function <- esse é o critério de avaliação
    inicio = time.time()
    rforest = sklearn.ensemble.RandomForestRegressor(n_estimators=individuo[0], criterion=individuo[1], max_depth=individuo[2],
        min_samples_split=individuo[3], min_samples_leaf=individuo[4], max_features=individuo[5])
    rforest.fit(x_train, y_train)
    eqm = np.sum((rforest.predict(x_train_validation) - y_train_validation)**2)
    tempo_execucao = time.time() - inicio
    custo = np.sqrt(eqm) + tempo_execucao # mude o fitness
    #custo = rforest.score(x_train_validation, y_train_validation) - tempo_execucao/50
    #print(individuo, np.round(custo, 3), np.round(tempo_execucao, 3))
    return custo,

def funcao_mutacao(individuo, indpb):
    individuo_novo = gerar_hiperparametros()
    for i in range(len(individuo)):
        if np.random.uniform(0, 1) < indpb:
            individuo[i] = individuo_novo[i]
    return individuo,

toolbox.register("mate", tools.cxOnePoint)
toolbox.register("mutate", funcao_mutacao, indpb=0.5)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", evaluate)

In [ ]:
pop = toolbox.population(n=100) # indivíduos na primeira geração

hof = deap.tools.HallOfFame(10) # objeto que contém os melhores indivíduos
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)
resultado = deap.algorithms.eaSimple(pop, toolbox, ngen=20, cxpb=1, mutpb=0.1, halloffame=hof, stats=stats)

gen	nevals	avg    	std    	min    	max    
0  	100   	71.6867	21.2347	43.4364	203.211
1  	100   	60.8813	10.1312	44.5231	98.5477
2  	100   	54.976 	13.8576	42.5472	169.065
3  	100   	48.8386	6.72501	42.4855	72.5981
4  	100   	46.5582	7.56955	42.242 	92.9706
5  	100   	45.2739	6.12003	42.3406	82.342 
6  	100   	46.0319	8.59284	41.9855	82.8467
7  	100   	44.0636	5.66783	42.171 	76.7907
8  	100   	44.442 	6.73153	42.1063	79.6434
9  	100   	44.7122	6.38753	41.5668	72.8223
10 	100   	44.9693	9.28572	41.775 	113.871
11 	100   	44.9823	7.61626	41.7081	83.6672
12 	100   	44.7704	7.08066	42.0129	77.4382
13 	100   	43.4315	4.40109	41.9697	77.0576
14 	100   	44.0269	5.36143	42.1978	73.1396
15 	100   	44.3366	5.94509	41.9724	76.1304
16 	100   	43.7534	4.19826	41.9643	66.1278
17 	100   	44.4357	6.45983	42.0793	80.2798
18 	100   	43.7991	5.24281	41.9546	70.8931
19 	100   	44.4015	6.49047	41.4362	83.3706
20 	100   	44.1141	5.67369	41.9465	77.8809


In [ ]:
for ind in hof: # melhores indivíduos em todas as gerações
    print(ind, evaluate(ind))

[30, 'squared_error', 32, 0.001, 0.001, 'log2'] (np.float64(42.90707048873246),)
[50, 'squared_error', 16, 0.001, 0.001, 'log2'] (np.float64(42.67953483736395),)
[50, 'friedman_mse', 32, 0.001, 0.001, 'log2'] (np.float64(42.71890329717864),)
[50, 'friedman_mse', 16, 0.001, 0.001, 'log2'] (np.float64(42.42368073863734),)
[30, 'squared_error', 16, 0.001, 0.001, 'log2'] (np.float64(42.920675457737076),)
[30, 'friedman_mse', 32, 0.001, 0.001, 'log2'] (np.float64(42.044105738836926),)
[50, 'squared_error', 32, 0.001, 0.001, 'log2'] (np.float64(42.70488328840663),)
[30, 'friedman_mse', 16, 0.001, 0.001, 'log2'] (np.float64(42.181809533131094),)
[100, 'friedman_mse', 32, 0.001, 0.001, 'log2'] (np.float64(43.04195532981517),)
[50, 'friedman_mse', 32, 0.001, 0.001, 'sqrt'] (np.float64(43.261866729621765),)


In [ ]:
for individuo in hof:
    rforest = sklearn.ensemble.RandomForestRegressor(n_estimators=individuo[0], criterion=individuo[1], max_depth=individuo[2],
        min_samples_split=individuo[3], min_samples_leaf=individuo[4], max_features=individuo[5])
    rforest.fit(x_test, y_test)
    eqm = np.sum((rforest.predict(x_test_validation) - y_test_validation)**2)
    print(np.sqrt(eqm))

29.29627857089468
28.974602118865185
29.52427336356245
29.31709195679988
29.28559353933248
29.074383871494412
29.39455091566333
29.37954881825736
29.046584721888866
29.876959331857144


In [ ]:
pop = toolbox.population(n=20) # indivíduos na primeira geração

hof = deap.tools.HallOfFame(10) # objeto que contém os melhores indivíduos
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)
resultado = deap.algorithms.eaMuCommaLambda(pop, toolbox, mu=90, lambda_=100, ngen=20, cxpb=0.7, mutpb=0.3, halloffame=hof, stats=stats)

gen	nevals	avg    	std    	min    	max    
0  	20    	78.5277	29.4157	46.7081	186.583
1  	100   	61.2559	9.10656	43.3483	84.0644
2  	100   	54.1982	7.6512 	43.2997	68.2549
3  	100   	49.3791	4.82709	42.9259	65.8445
4  	100   	46.842 	3.20903	42.4599	58.1749
5  	100   	44.7219	1.69875	42.3042	49.3304
6  	100   	43.8902	1.17829	42.3958	47.855 
7  	100   	42.8723	0.647707	42.1027	45.3943
8  	100   	42.9044	1.6644  	42.1142	57.7854
9  	100   	42.7964	2.65462 	41.6404	67.524 
10 	100   	42.7317	2.07572 	41.7384	62.1273
11 	100   	42.5469	0.268201	41.9471	43.1423
12 	100   	43.164 	4.67581 	41.9972	76.6555
13 	100   	42.9883	1.83811 	41.4505	58.223 
14 	100   	43.0104	2.26949 	41.9658	57.3936
15 	100   	42.8235	1.6285  	41.9784	57.8385
16 	100   	42.6994	1.78298 	41.8639	58.9927
17 	100   	42.5069	0.593641	41.8146	46.6748
18 	100   	42.7275	2.40251 	42.051 	64.81  
19 	100   	42.8082	2.63842 	42.0963	67.5801
20 	100   	42.619 	0.461604	42.1456	45.8594


In [ ]:
for ind in hof: # melhores indivíduos em todas as gerações
    print(ind, evaluate(ind))

[30, 'poisson', 16, 0.001, 0.001, 'log2'] (np.float64(43.121107268942765),)
[50, 'poisson', 16, 0.001, 0.001, 'log2'] (np.float64(43.688239565234134),)
[50, 'squared_error', 16, 0.001, 0.001, 'log2'] (np.float64(42.96346882755365),)
[30, 'friedman_mse', 16, 0.001, 0.001, 'log2'] (np.float64(43.33123940008176),)
[50, 'squared_error', 32, 0.001, 0.001, 'log2'] (np.float64(43.17898074161268),)
[30, 'squared_error', 32, 0.001, 0.001, 'log2'] (np.float64(43.13829106342625),)
[50, 'poisson', 32, 0.001, 0.001, 'log2'] (np.float64(42.81659144771325),)
[30, 'poisson', 32, 0.001, 0.001, 'log2'] (np.float64(42.651027708107456),)
[50, 'friedman_mse', 16, 0.001, 0.001, 'log2'] (np.float64(42.32064636865263),)
[30, 'squared_error', 16, 0.001, 0.001, 'log2'] (np.float64(42.705795820626705),)


In [ ]:
for individuo in hof:
    rforest = sklearn.ensemble.RandomForestRegressor(n_estimators=individuo[0], criterion=individuo[1], max_depth=individuo[2],
        min_samples_split=individuo[3], min_samples_leaf=individuo[4], max_features=individuo[5])
    rforest.fit(x_test, y_test)
    eqm = np.sum((rforest.predict(x_test_validation) - y_test_validation)**2)
    print(eqm)

860.4466182484434
847.6118764952073
870.3637410860251
849.0007025562518
875.9199471872789
866.3997376850565
859.742214768811
903.0556845223824
866.978790954094
878.8687720678872


In [ ]:
for ind in hof: # melhores indivíduos em todas as gerações
    evaluate(ind)

In [ ]:
for individuo in hof:
    rforest = sklearn.ensemble.RandomForestRegressor(n_estimators=individuo[0], criterion=individuo[1], max_depth=individuo[2],
        min_samples_split=individuo[3], min_samples_leaf=individuo[4], max_features=individuo[5])
    rforest.fit(x_test, y_test)
    eqm = np.sum((rforest.predict(x_test_validation) - y_test_validation)**2)
    print(eqm)

876.2423033627521
862.048216578439
844.8593378687598
830.3670369985692
874.236110939482
834.252958805475
873.8120999405675
890.7892131272754
867.6322700670426
876.9566698580038
